# 04 — Main model and robustness checks

Everything here follows `PREREGISTRATION.md` (sections 3–5), committed before any ghost-vs-crowd comparison.

Main model (OLS, one row per match, 2017/18–2022/23):

```
card_diff ~ ghost + partial + elo_diff + var + trend + C(Referee)
cpf_diff  ~ ghost + partial + elo_diff + var + trend + C(Referee)      (low_fouls == 0)
```

- standard errors clustered by referee;
- primary p-value from a wild cluster bootstrap (Rademacher weights, 9,999 draws, null imposed, seed 2026);
- minimum detectable effect MDE = 2.8 × clustered SE.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from wildboottest.wildboottest import wildboottest

PROCESSED_DIR = Path("../data/processed")
pd.set_option("display.width", 200)

matches = pd.read_csv(PROCESSED_DIR / "matches_elo.csv", parse_dates=["Date"])
analysis = matches[matches["season_start"] >= 2017].copy()
print(analysis.shape)

(2280, 28)


## 1. One function for every OLS estimate

`fit_ghost_model` fits the model, clusters by referee, runs the wild bootstrap and returns one row with everything we report about the `ghost` coefficient.

Rows with a missing outcome are removed **before** fitting, so the bootstrap's cluster list lines up exactly with the rows the model uses.

In [2]:
BOOTSTRAP_DRAWS = 9999
SEED = 2026
MDE_MULTIPLIER = 2.8          # 80% power, alpha = 0.05, two-sided


def fit_ghost_model(data, outcome, right_side, label, param="ghost"):
    data = data.dropna(subset=[outcome]).reset_index(drop=True)
    referee_ids = pd.factorize(data["Referee"])[0]

    model = smf.ols(f"{outcome} ~ {right_side}", data=data)
    fitted = model.fit(cov_type="cluster", cov_kwds={"groups": referee_ids})

    boot = wildboottest(model, param=param, cluster=referee_ids, B=BOOTSTRAP_DRAWS,
                        weights_type="rademacher", impose_null=True, seed=SEED, show=False)

    se = fitted.bse[param]
    return {
        "check": label,
        "outcome": outcome,
        "coef": fitted.params[param],
        "se_cluster": se,
        "ci_low": fitted.conf_int().loc[param, 0],
        "ci_high": fitted.conf_int().loc[param, 1],
        "p_cluster": fitted.pvalues[param],
        "p_bootstrap": float(boot["p-value"].iloc[0]),
        "mde": MDE_MULTIPLIER * se,
        "matches": len(data),
        "ghost_matches": int(data["ghost"].sum()),
        "referees": data["Referee"].nunique(),
    }


MAIN_RIGHT_SIDE = "ghost + partial + elo_diff + var + trend + C(Referee)"

## 2. Main model

`cpf_diff` is already empty for the 28 `low_fouls` matches (set in notebook 01), so `dropna` applies the pre-registered exclusion.

In [3]:
main = pd.DataFrame([
    fit_ghost_model(analysis, "card_diff", MAIN_RIGHT_SIDE, "main"),
    fit_ghost_model(analysis, "cpf_diff", MAIN_RIGHT_SIDE, "main"),
])

# Holm correction across the two hypotheses (H1, H1b), on the primary bootstrap p-values
main["p_bootstrap_holm"] = multipletests(main["p_bootstrap"], method="holm")[1]
main.round(4).T

,0,1
check,main,main
outcome,card_diff,cpf_diff
coef,-0.225,-0.0082
se_cluster,0.0747,0.0084
ci_low,-0.3714,-0.0247
ci_high,-0.0787,0.0083
p_cluster,0.0026,0.3287
p_bootstrap,0.0089,0.3777
mde,0.2091,0.0236
matches,2280,2265


In [4]:
# Full output of the main card_diff model (referee dummies hidden) — for reference
data = analysis.dropna(subset=["card_diff"]).reset_index(drop=True)
full_fit = smf.ols(f"card_diff ~ {MAIN_RIGHT_SIDE}", data=data).fit(
    cov_type="cluster", cov_kwds={"groups": pd.factorize(data["Referee"])[0]})
summary = pd.DataFrame({"coef": full_fit.params, "se": full_fit.bse, "p": full_fit.pvalues})
summary[~summary.index.str.startswith("C(Referee)")].round(4)

,coef,se,p
Intercept,0.7811,0.3662,0.0329
ghost,-0.2250,0.0747,0.0026
partial,0.3272,0.2161,0.1301
elo_diff,0.0016,0.0002,0.0000
var,0.0895,0.1171,0.4448
trend,-0.0471,0.0396,0.2343


## 3. Robustness checks (OLS)

1. Season fixed effects instead of `var + trend`. The effect is then identified mainly from the 2019/20 restart, because 2020/21 is almost entirely ghost and its season dummy absorbs it.
2. Partial matches excluded.
3. VAR seasons only (2019/20–2022/23). `var` is constant there, so it is dropped.
4. 2020/21 ghost only: the 92 ghost matches of the 2019/20 restart are excluded.

In [5]:
SEASON_FE_RIGHT_SIDE = "ghost + partial + elo_diff + C(season) + C(Referee)"
VAR_ONLY_RIGHT_SIDE = "ghost + partial + elo_diff + trend + C(Referee)"
NO_PARTIAL_RIGHT_SIDE = "ghost + elo_diff + var + trend + C(Referee)"

no_partial = analysis[analysis["partial"] == 0]
var_seasons = analysis[analysis["season_start"] >= 2019]
without_restart = analysis[~((analysis["season_start"] == 2019) & (analysis["ghost"] == 1))]

checks = []
for outcome in ["card_diff", "cpf_diff"]:
    checks.append(fit_ghost_model(analysis, outcome, SEASON_FE_RIGHT_SIDE, "1. season FE"))
    checks.append(fit_ghost_model(no_partial, outcome, NO_PARTIAL_RIGHT_SIDE, "2. no partial"))
    checks.append(fit_ghost_model(var_seasons, outcome, VAR_ONLY_RIGHT_SIDE, "3. VAR seasons only"))
    checks.append(fit_ghost_model(without_restart, outcome, MAIN_RIGHT_SIDE, "4. 2020/21 ghost only"))

robustness = pd.concat([main.drop(columns="p_bootstrap_holm"), pd.DataFrame(checks)], ignore_index=True)
robustness = robustness.sort_values(["outcome", "check"]).reset_index(drop=True)
robustness[["outcome", "check", "coef", "ci_low", "ci_high", "p_cluster", "p_bootstrap", "mde",
            "matches", "ghost_matches", "referees"]].round(4)

,outcome,check,coef,ci_low,ci_high,p_cluster,p_bootstrap,mde,matches,ghost_matches,referees
0,card_diff,1. season FE,-0.2712,-0.6434,0.1010,0.1533,0.1885,0.5317,2280,437,33
1,card_diff,2. no partial,-0.2247,-0.3713,-0.0781,0.0027,0.0092,0.2094,2245,437,33
2,card_diff,3. VAR seasons only,-0.2370,-0.3892,-0.0849,0.0023,0.0093,0.2174,1520,437,29
3,card_diff,4. 2020/21 ghost only,-0.2257,-0.3777,-0.0737,0.0036,0.0089,0.2171,2188,345,33
4,card_diff,main,-0.2250,-0.3714,-0.0787,0.0026,0.0089,0.2091,2280,437,33
5,cpf_diff,1. season FE,-0.0252,-0.0593,0.0090,0.1483,0.1787,0.0487,2265,435,33
6,cpf_diff,2. no partial,-0.0082,-0.0247,0.0084,0.3331,0.3816,0.0236,2231,435,33
7,cpf_diff,3. VAR seasons only,-0.0097,-0.0270,0.0075,0.2686,0.2899,0.0246,1509,435,29
8,cpf_diff,4. 2020/21 ghost only,-0.0054,-0.0225,0.0116,0.5332,0.5756,0.0244,2173,343,33
9,cpf_diff,main,-0.0082,-0.0247,0.0083,0.3287,0.3777,0.0236,2265,435,33


## 4. Robustness check 5: Poisson model of cards per foul

A third way to condition cards on fouls (next to the ratio difference and fouls-as-control in the literature).

- **Data shape:** one row per team per match (two rows per match).
- **Model:** yellow cards are modelled with a Poisson regression, with `log(fouls)` as an *offset*. An offset is a variable whose coefficient is fixed at 1, so the model describes the **rate of yellows per foul**.
- **Coefficient of interest:** `away:ghost`, the change in the away team's cards-per-foul rate (relative to the home team's) when there is no crowd. `exp(coef) − 1` is the percentage change.
- **Sample:** `low_fouls` matches excluded; SEs clustered by referee. The wild bootstrap package only supports OLS, so here we report the clustered p-value only.

In [6]:
base = analysis[analysis["low_fouls"] == 0]
shared = ["Referee", "ghost", "partial", "var", "trend", "season_start"]

home_rows = base[shared].assign(away=0, yellows=base["HY"], fouls=base["HF"], team_elo_diff=base["elo_diff"])
away_rows = base[shared].assign(away=1, yellows=base["AY"], fouls=base["AF"], team_elo_diff=-base["elo_diff"])
team_match = pd.concat([home_rows, away_rows], ignore_index=True)
print(team_match.shape)
team_match.head()

(4530, 10)


,Referee,ghost,partial,var,trend,season_start,away,yellows,fouls,team_elo_diff
0,M Dean,0,0,0,7,2017,0,0,9,132.547090
1,M Oliver,0,0,0,7,2017,0,0,6,-299.909216
2,C Pawson,0,0,0,7,2017,0,3,16,306.346704
3,J Moss,0,0,0,7,2017,0,1,7,76.180943
4,N Swarbrick,0,0,0,7,2017,0,1,13,63.927343


In [7]:
poisson = smf.glm(
    "yellows ~ away + away:ghost + away:partial + ghost + partial + team_elo_diff + var + trend + C(Referee)",
    data=team_match,
    family=sm.families.Poisson(),
    offset=np.log(team_match["fouls"]),
).fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(team_match["Referee"])[0]})

rows = ["away", "away:ghost", "away:partial", "ghost", "partial", "team_elo_diff"]
poisson_table = pd.DataFrame({
    "coef": poisson.params[rows],
    "se_cluster": poisson.bse[rows],
    "p_cluster": poisson.pvalues[rows],
    "rate_ratio": np.exp(poisson.params[rows]),
})
poisson_table.round(4)

,coef,se_cluster,p_cluster,rate_ratio
away,0.0745,0.0188,0.0001,1.0773
away:ghost,-0.0405,0.0329,0.2179,0.9603
away:partial,0.3218,0.1719,0.0612,1.3796
ghost,-0.2310,0.0435,0.0000,0.7937
partial,-0.2652,0.1641,0.1061,0.7671
team_elo_diff,-0.0005,0.0000,0.0000,0.9995


In [8]:
coef = poisson.params["away:ghost"]
low, high = poisson.conf_int().loc["away:ghost"]
print(f"away x ghost: {np.exp(coef) - 1:+.1%} change in away teams' cards per foul relative to home teams "
      f"(95% CI {np.exp(low) - 1:+.1%} to {np.exp(high) - 1:+.1%}), p = {poisson.pvalues['away:ghost']:.3f}")
print(f"For reference, with crowds away teams' cards-per-foul rate is {np.exp(poisson.params['away']) - 1:+.1%} vs home teams")

away x ghost: -4.0% change in away teams' cards per foul relative to home teams (95% CI -10.0% to +2.4%), p = 0.218
For reference, with crowds away teams' cards-per-foul rate is +7.7% vs home teams


## 5. Save results

In [9]:
robustness.to_csv(PROCESSED_DIR / "results_ols.csv", index=False)
poisson_table.to_csv(PROCESSED_DIR / "results_poisson.csv")
main.to_csv(PROCESSED_DIR / "results_main.csv", index=False)
print("Saved results to data/processed/")

Saved results to data/processed/
